# Build one mosaic: pick a cell, pick a year, export

This notebook builds and exports the annual Landsat mosaic for one grid cell
(one 1:250,000 map sheet of India) and one or more years.

It uses exactly the same pipeline code as the batch driver
(`pipeline/run.py`), so what you see here is what a full run produces.

A "year" is the phenological year: 1 April of the labelled year to 31 March
of the next. So `2019` means April 2019 to March 2020.

Three steps: set the parameters below, run the quick-look cell to preview the
mosaic, then run the export cell to queue the full asset. For many cells or
many years, the batch driver (`python -m pipeline.run --export`) is the
reliable path -- it resumes cleanly and skips work already done.

In [ ]:
# ---------------------------------------------------------------------------
# Parameters -- the only cell you normally edit
# ---------------------------------------------------------------------------

CELL = 'NC-43-X-D'          # grid cell name (must exist in the CIM grid asset)
YEARS = [2019]              # one or more phenological years, e.g. [2018, 2019]

# Where the exported mosaic lands:
#   'development' -> the sandbox collection (default; safe to write to)
#   'production'  -> the published MapBiomas collection (write access needed)
DESTINATION = 'development'

In [ ]:
# ---------------------------------------------------------------------------
# Setup: Earth Engine, pipeline imports, destination path
# ---------------------------------------------------------------------------

import os
import sys

# The notebook lives in notebooks/; the pipeline package lives one level up.
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if os.path.isdir(os.path.join(repo_root, 'pipeline')) and repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import ee

from pipeline import config
from pipeline import build
from pipeline import run

ee.Initialize(project=config.EE_PROJECT)

# Same check the batch driver runs: warns loudly about any coefficient set
# not yet marked verified. Warnings do not block a run.
run.preflight()

if DESTINATION == 'development':
    COLLECTION_PATH = config.SANDBOX_COLLECTION
elif DESTINATION == 'production':
    COLLECTION_PATH = config.PRODUCTION_COLLECTION
else:
    raise ValueError("DESTINATION must be 'development' or 'production', "
                     'got {!r}'.format(DESTINATION))

print('\nexports will go to: {}'.format(COLLECTION_PATH))

In [ ]:
# ---------------------------------------------------------------------------
# Build + quick look
# ---------------------------------------------------------------------------
# Assembles the mosaic for the first year in YEARS (nothing is computed on
# the server until the thumbnail below pulls on it), then renders a small
# false-colour preview: shortwave infrared / near infrared / red, the usual
# way of looking at these mosaics. Values are reflectance x 10000.

from IPython.display import Image as IPyImage

preview_year = YEARS[0]
mosaic, meta = build.build_mosaic(CELL, preview_year, variant='c2_only')

region = build.cell_geometry(CELL)
thumb_url = mosaic.select(['swir1_median', 'nir_median', 'red_median']).getThumbURL({
    'region': region,
    'dimensions': 512,
    'min': 0,
    'max': 4000,
})

print('preview of {} {} (a reduced-scale render, not the export itself):'
      .format(CELL, preview_year))
IPyImage(url=thumb_url)

In [ ]:
# ---------------------------------------------------------------------------
# Export: queue one task per year
# ---------------------------------------------------------------------------
# Mirrors the batch driver's call. Each task builds the full mosaic at 30 m
# and writes it into the collection chosen above. Already-exported years are
# skipped, so re-running this cell is safe.

tasks = []
for year in YEARS:
    task = build.export(CELL, year, variant='c2_only',
                        collection_path=COLLECTION_PATH, verbose=True)
    if task is not None:
        tasks.append(task)

print('\n{} task(s) queued to {}'.format(len(tasks), COLLECTION_PATH))
for task in tasks:
    print('  {}'.format(task.config['description']))

## What happens next

Each queued task runs on Google's servers -- you can close this notebook.
Watch progress in the **Tasks** tab of the [Earth Engine Code Editor](https://code.earthengine.google.com/)
(or with `earthengine task list`). A full cell-year typically takes a while;
a task that fails reports its error there too.

When a task finishes, the mosaic appears inside the collection printed
above. The image name depends on the destination: the development
sandbox (`.../shared_assets/ioln_mosaics_v2_sandbox`)
uses `<CELL>_<year>_c2_only_v<version>`; the production collection
uses `<CELL>_<year>` only (the product version is stored in the
image properties instead). Years with no usable
satellite imagery (common before 2000 in some regions) are skipped with a
message rather than exported empty: the gap is real and the honest product
is to show it.